# SBA loan default prediction project  — Scoring Function

This notebook implements `project_1_scoring()` for both model families.  
It loads saved artifacts, applies the correct preprocessing pipeline,  
and returns predictions in the required format.

In [13]:
# ── Scoring notebook: Imports & H2O init ─────────────────────────────────
import pandas as pd
import numpy as np
import joblib
import os
import h2o

h2o.init(max_mem_size="4G")

Checking whether there is an H2O instance running at http://localhost:54321. connected.
Please download and install the latest version from: https://h2o-release.s3.amazonaws.com/h2o/latest_stable.html


H2O_cluster_uptime:,2 hours 4 mins
H2O_cluster_timezone:,America/Chicago
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.9
H2O_cluster_version_age:,3 months and 21 days
H2O_cluster_name:,H2O_from_python_bassi_kr3qk3
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,3.551 Gb
H2O_cluster_total_cores:,8
H2O_cluster_allowed_cores:,8
H2O_cluster_status:,"locked, healthy"


In [14]:
# ── project_1_scoring: main scoring function ──────────────────────────────

def project_1_scoring(data, model_type="sklearn"):
    """
    Score new SBA loan records using trained models.

    Input:
        data       : Pandas DataFrame — same schema as project data,
                     WITHOUT the MIS_Status column
        model_type : 'sklearn' or 'h2o'

    Output:
        Pandas DataFrame with columns:
            index         — original DataFrame index
            label         — predicted class (0 or 1)
            probability_0 — probability of non-default
            probability_1 — probability of default
    """

    df = data.copy()

    # ── Shared preprocessing (same for both model families) ───────────────

    # 1. Drop columns not used in training
    drop_cols = ["City", "Bank", "BankState", "Zip", "BalanceGross",
                 "MIS_Status", "index"]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])

    # 2. Fix NAICS — replace 0 with NaN then fill back to 0
    if "NAICS" in df.columns:
        df["NAICS"] = df["NAICS"].replace(0, np.nan).fillna(0).astype(int)
        df["NAICS_sector"] = (df["NAICS"] // 10000).astype(int)
        df = df.drop(columns=["NAICS"])

    # 3. Fix State
    if "State" in df.columns:
        df["State"] = df["State"].fillna("Unknown")

    # 4. Fix NewExist
    if "NewExist" in df.columns:
        df["NewExist"] = df["NewExist"].replace(0.0, 1.0).fillna(1.0).astype(int)

    # 5. Standardize RevLineCr and LowDoc
    valid_map = {"Y": 1, "1": 1, "N": 0, "0": 0, 1: 1, 0: 0}
    for col in ["RevLineCr", "LowDoc"]:
        if col in df.columns:
            df[col] = df[col].map(valid_map).fillna(0).astype(int)

    # 6. FranchiseCode → is_franchise
    if "FranchiseCode" in df.columns:
        df["is_franchise"] = (df["FranchiseCode"] > 1).astype(int)
        df = df.drop(columns=["FranchiseCode"])

    # 7. Drop DisbursementGross == 0 rows would drop records — skip in scoring
    #    Instead clip to a small value to avoid division issues
    if "DisbursementGross" in df.columns:
        df["DisbursementGross"] = df["DisbursementGross"].clip(lower=0.01)

    # ── Load model-specific artifacts ─────────────────────────────────────
    if model_type == "sklearn":
        arts = joblib.load("artifacts/sklearn/sklearn_artifacts.pkl")
    elif model_type == "h2o":
        arts = joblib.load("artifacts/h2o/h2o_artifacts.pkl")
    else:
        raise ValueError("model_type must be 'sklearn' or 'h2o'")

    te_encoder  = arts["te_encoder"]
    woe_encoder = arts["woe_encoder"]
    ohe_cols    = arts["ohe_cols"]
    te_cols     = arts["te_cols"]
    threshold   = arts["optimal_threshold"]

    # ── OHE (replicate training dummies) ──────────────────────────────────
    df = pd.get_dummies(df, columns=ohe_cols, prefix=ohe_cols)

    # ── Target encoding (transform only — no fit) ─────────────────────────
    for col in te_cols:
        if col in df.columns:
            df[f"{col}_trg"] = te_encoder.transform(df[te_cols])[col]
    df = df.drop(columns=[c for c in te_cols if c in df.columns])

    # ── Feature engineering ───────────────────────────────────────────────
    df["sba_coverage_ratio"]    = df["SBA_Appv"] / (df["GrAppv"] + 1e-6)
    df["loan_per_employee"]     = df["GrAppv"] / (df["NoEmp"] + 1)
    df["disbursement_ratio"]    = df["DisbursementGross"] / (df["GrAppv"] + 1e-6)
    df["job_creation_ratio"]    = df["CreateJob"] / (df["NoEmp"] + 1)
    df["log_GrAppv"]            = np.log1p(df["GrAppv"])
    df["log_NoEmp"]             = np.log1p(df["NoEmp"])
    df["log_DisbursementGross"] = np.log1p(df["DisbursementGross"])
    df["retained_to_emp_ratio"] = df["RetainedJob"] / (df["NoEmp"] + 1)
    df["is_low_doc_franchise"]  = df["LowDoc"] * df["is_franchise"]
    df["sba_appv_per_emp"]      = df["SBA_Appv"] / (df["NoEmp"] + 1)
    df["total_jobs"]            = df["CreateJob"] + df["RetainedJob"]
    df["GrAppv_bin"]            = pd.qcut(df["GrAppv"], q=5,
                                          labels=False, duplicates="drop")
    df["GrAppv_bin_woe"]        = woe_encoder.transform(
                                      df[["GrAppv_bin"]])["GrAppv_bin"]
    df = df.drop(columns=["GrAppv_bin"])

    # ── Align columns to training schema ──────────────────────────────────
    if model_type == "sklearn":
        train_cols = arts["train_columns"]
        scaler     = arts["scaler"]
        sc_cols    = arts["scaler_cols"]
        model      = arts["model"]

        df = df.reindex(columns=train_cols, fill_value=0)
        df[sc_cols] = scaler.transform(df[sc_cols])

        probs  = model.predict_proba(df)
        prob_0 = probs[:, 0]
        prob_1 = probs[:, 1]

    elif model_type == "h2o":
        feature_cols = arts["feature_cols"]
        df = df.reindex(columns=feature_cols, fill_value=0)

        # Load H2O model — find the model file automatically
        h2o_model_path = [
            os.path.join("artifacts/h2o", f)
            for f in os.listdir("artifacts/h2o")
            if not f.endswith(".pkl") and not f.endswith(".gitkeep")
        ][0]
        h2o_model = h2o.load_model(h2o_model_path)

        hf     = h2o.H2OFrame(df)
        preds  = h2o_model.predict(hf).as_data_frame()
        prob_0 = preds["p0"].values
        prob_1 = preds["p1"].values

    # ── Assemble output DataFrame ──────────────────────────────────────────
    labels = (prob_1 >= threshold).astype(int)

    output = pd.DataFrame({
        "index"        : data.index,
        "label"        : labels,
        "probability_0": prob_0,
        "probability_1": prob_1
    })

    return output

In [15]:
# ── Validate scoring function end-to-end ──────────────────────────────────

# Simulate new data (test set without target column)
new_data = pd.read_csv("data/SBA_loans_project_1.csv.zip").sample(
    500, random_state=99
)

# Remove target column — scoring function must not see it
new_data_no_target = new_data.drop(columns=["MIS_Status"])

print("Testing sklearn scoring...")
sklearn_out = project_1_scoring(new_data_no_target, model_type="sklearn")
print(f"  Output shape : {sklearn_out.shape}")
print(f"  Columns      : {sklearn_out.columns.tolist()}")
print(f"  Null values  : {sklearn_out.isna().sum().sum()}")
print(f"  Records in   : {len(new_data_no_target)} | Records out: {len(sklearn_out)}")
print(f"  Label counts : {sklearn_out['label'].value_counts().to_dict()}")
print(sklearn_out.head(3))

print("\nTesting H2O scoring...")
h2o_out = project_1_scoring(new_data_no_target, model_type="h2o")
print(f"  Output shape : {h2o_out.shape}")
print(f"  Null values  : {h2o_out.isna().sum().sum()}")
print(f"  Records in   : {len(new_data_no_target)} | Records out: {len(h2o_out)}")
print(f"  Label counts : {h2o_out['label'].value_counts().to_dict()}")
print(h2o_out.head(3))

print("\n Scoring function validated — both model types working correctly")

Testing sklearn scoring...
  Output shape : (500, 4)
  Columns      : ['index', 'label', 'probability_0', 'probability_1']
  Null values  : 0
  Records in   : 500 | Records out: 500
  Label counts : {0: 395, 1: 105}
    index  label  probability_0  probability_1
0  666409      0       0.804625       0.195375
1   80822      0       0.824372       0.175628
2  246033      0       0.764749       0.235251

Testing H2O scoring...
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
glm prediction progress: |███████████████████████████████████████████████████████| (done) 100%


c:\Users\bassi\BUAN-6341-ML\learning-modules-bass990\.venv\Lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


  Output shape : (500, 4)
  Null values  : 0
  Records in   : 500 | Records out: 500
  Label counts : {0: 396, 1: 104}
    index  label  probability_0  probability_1
0  666409      0       0.805354       0.194646
1   80822      0       0.821526       0.178474
2  246033      0       0.764592       0.235408

 Scoring function validated — both model types working correctly


In [16]:
# ── Final validation on official holdout dataset ──────────────────────────
import pandas as pd
holdout = pd.read_csv("data/SBA_loans_project_1_holdout_students_valid_no_labels.csv.zip")

print(f"Holdout shape      : {holdout.shape}")
print(f"Holdout columns    : {holdout.columns.tolist()}")
print(f"MIS_Status present : {'MIS_Status' in holdout.columns}")  # must be False

print("\nTesting sklearn on holdout...")
holdout_sklearn = project_1_scoring(holdout, model_type="sklearn")
print(f"  Records in  : {len(holdout)}")
print(f"  Records out : {len(holdout_sklearn)}")
print(f"  Null values : {holdout_sklearn.isna().sum().sum()}")
print(f"  Label counts: {holdout_sklearn['label'].value_counts().to_dict()}")
print(holdout_sklearn.head(3))

print("\nTesting H2O on holdout...")
holdout_h2o = project_1_scoring(holdout, model_type="h2o")
print(f"  Records in  : {len(holdout)}")
print(f"  Records out : {len(holdout_h2o)}")
print(f"  Null values : {holdout_h2o.isna().sum().sum()}")
print(f"  Label counts: {holdout_h2o['label'].value_counts().to_dict()}")
print(holdout_h2o.head(3))

print("\n Holdout scoring validated — ready for submission")

Holdout shape      : (89917, 19)
Holdout columns    : ['index', 'City', 'State', 'Zip', 'Bank', 'BankState', 'NAICS', 'NoEmp', 'NewExist', 'CreateJob', 'RetainedJob', 'FranchiseCode', 'UrbanRural', 'RevLineCr', 'LowDoc', 'DisbursementGross', 'BalanceGross', 'GrAppv', 'SBA_Appv']
MIS_Status present : False

Testing sklearn on holdout...
  Records in  : 89917
  Records out : 89917
  Null values : 0
  Label counts: {0: 72570, 1: 17347}
   index  label  probability_0  probability_1
0      0      0       0.833668       0.166332
1      1      0       0.842401       0.157599
2      2      0       0.785976       0.214024

Testing H2O on holdout...
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
glm prediction progress: |███████████████████████████████████████████████████████| (done) 100%


c:\Users\bassi\BUAN-6341-ML\learning-modules-bass990\.venv\Lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


  Records in  : 89917
  Records out : 89917
  Null values : 0
  Label counts: {0: 72564, 1: 17353}
   index  label  probability_0  probability_1
0      0      0       0.832722       0.167278
1      1      0       0.843148       0.156852
2      2      0       0.784699       0.215301

 Holdout scoring validated — ready for submission


In [22]:
# ── Kaggle submissions — fixed column names ───────────────────────────────

# H2O GLM
submission_h2o = pd.DataFrame({
    "ID"           : holdout_h2o["index"],
    "probability_1": holdout_h2o["probability_1"]
})
submission_h2o.to_csv("predictions_h2o.csv", index=False)
print("H2O submission fixed ")
print(f"  Columns: {submission_h2o.columns.tolist()}")
print(f"  Shape  : {submission_h2o.shape}")

# sklearn LR
submission_sklearn = pd.DataFrame({
    "ID"           : holdout_sklearn["index"],
    "probability_1": holdout_sklearn["probability_1"]
})
submission_sklearn.to_csv("predictions_sklearn.csv", index=False)
print("\nsklearn submission fixed ")
print(f"  Columns: {submission_sklearn.columns.tolist()}")
print(f"  Shape  : {submission_sklearn.shape}")

print("\nPreview H2O:")
print(submission_h2o.head(3))
print("\nPreview sklearn:")
print(submission_sklearn.head(3))



H2O submission fixed 
  Columns: ['ID', 'probability_1']
  Shape  : (89917, 2)

sklearn submission fixed 
  Columns: ['ID', 'probability_1']
  Shape  : (89917, 2)

Preview H2O:
   ID  probability_1
0   0       0.167278
1   1       0.156852
2   2       0.215301

Preview sklearn:
   ID  probability_1
0   0       0.166332
1   1       0.157599
2   2       0.214024


In [25]:
# Check holdout index values
print("Holdout 'index' column sample:")
print(holdout["index"].head(10).tolist())
print("\nHoldout index range:", holdout["index"].min(), "–", holdout["index"].max())

# Check what we submitted
check = pd.read_csv("predictions_h2o.csv")
print("\nSubmission 'id' column sample:")
print(check["ID"].head(10).tolist())
print("\nSubmission id range:", check["ID"].min(), "–", check["ID"].max())

# Check if sorted correctly
print("\nIs submission sorted by id?", check["ID"].is_monotonic_increasing)

Holdout 'index' column sample:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

Holdout index range: 0 – 89916

Submission 'id' column sample:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

Submission id range: 0 – 89916

Is submission sorted by id? True
